# Priority 2 — Fixed Ablation Study

Runs ablation on **adversarial** and **anisotropic** datasets — the two datasets where components actually differentiate.

Previous ablation ran on `correlated` where all variants scored 0.998, making it impossible to assess individual component contributions.

**Variants tested:**
- Deep-MELU (full)
- No t-CDF (MELU-base: ν→∞, Term 1 becomes plain Swish)
- No MCD (Euclidean whitening: L_inv = I)
- No L_contrastive (λ₁=0)
- No amplifier (α=0: removes Term 2 entirely)
- No APN-MCD (use BatchNorm-style normalisation)
- Ablate all (vanilla autoencoder baseline)

## Cell 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.special import betainc
from scipy.stats import chi2
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)
print("Imports OK")


## Cell 2 — Dataset generators

Both datasets are fully deterministic given a seed.

In [ ]:
def make_adversarial(n=500, dim=8, contamination=0.12, seed=42):
    """
    Outliers placed just outside the 95% inlier ellipse.
    This is the hardest dataset — small margin, no easy separation.
    """
    np.random.seed(seed)
    n_out = max(1, int(n * contamination)); n_in = n - n_out
    X_in  = np.random.randn(n_in, dim)
    r95   = np.sqrt(chi2.ppf(0.95, df=dim))
    X_out = np.array([
        np.random.randn(dim) / np.linalg.norm(np.random.randn(dim)) *
        r95 * (1.05 + np.random.rand() * 0.30)
        for _ in range(n_out)])
    X = np.vstack([X_in, X_out])
    y = np.array([0]*n_in + [1]*n_out)
    return X, y

def make_anisotropic(n=500, dim=8, contamination=0.10, seed=42):
    """
    Variance spans 3 orders of magnitude (σ: 0.1 → 10).
    Outliers are extreme in the single lowest-variance dimension.
    Euclidean distance is blind to this; Mahalanobis is not.
    """
    np.random.seed(seed)
    n_out  = max(1, int(n * contamination)); n_in = n - n_out
    scales = np.logspace(-1, 1, dim)   # σ_0=0.1, σ_{dim-1}=10
    X_in   = np.random.randn(n_in, dim) * scales
    X_out  = np.random.randn(n_out, dim) * scales
    low_d  = np.argmin(scales)
    X_out[:, low_d] += np.random.choice([-1,1], n_out) * scales[low_d] * 5
    X = np.vstack([X_in, X_out])
    y = np.array([0]*n_in + [1]*n_out)
    return X, y

X_adv, y_adv = make_adversarial()
X_ani, y_ani = make_anisotropic()

for name, (X, y) in [("adversarial", (X_adv, y_adv)),
                      ("anisotropic", (X_ani, y_ani))]:
    n_out = int(y.sum())
    print(f"{name:15s}  n={len(X)}  dim={X.shape[1]}  "
          f"outliers={n_out}  ({n_out/len(X)*100:.1f}%)")


## Cell 3 — Ablation model

Each variant is created by modifying one parameter of the base `DeepMELU` class.  
`nu=1e6` collapses T_nu → sigmoid (≈ Swish)  
`Li=I` removes Mahalanobis whitening  
`alpha=0` removes the amplifier entirely  
`lam1=0` removes the contrastive loss

In [ ]:
def _tcdf(x, nu=4.0):
    nu = max(float(nu), 2.0)
    z  = nu / (nu + np.clip(x**2, 1e-30, None))
    ib = betainc(nu/2, 0.5, np.clip(z, 1e-12, 1-1e-12))
    return np.where(x >= 0, 1.0-ib/2.0, ib/2.0)

class AblationModel:
    """
    Deep-MELU with configurable ablation flags.

    Params
    ------
    use_tcdf        : if False, nu=1e6 => T_nu → sigmoid (Swish-like base)
    use_mcd         : if False, L_inv = I (Euclidean whitening)
    use_amplifier   : if False, alpha=0 (no exponential amplifier)
    use_contrastive : if False, lam1=0 (no contrastive loss)
    use_apn_norm    : if False, skip whitening (use standard BatchNorm-style centering)
    """
    def __init__(self, dim, latent=32, hidden=64,
                 use_tcdf=True, use_mcd=True,
                 use_amplifier=True, use_contrastive=True,
                 use_apn_norm=True,
                 alpha=1.0, beta=0.4, nu=4.0, momentum=0.95):
        self.dim = dim; self.latent = latent
        self.use_tcdf = use_tcdf
        self.use_mcd  = use_mcd
        self.use_amp  = use_amplifier
        self.use_cont = use_contrastive
        self.use_apn  = use_apn_norm
        self.alpha = alpha if use_amplifier else 0.0
        self.beta  = beta
        self.nu    = nu if use_tcdf else 1e6
        self.momentum = momentum
        np.random.seed(0); s = np.sqrt(2/dim)
        self.W1 = np.random.randn(dim,    hidden) * s
        self.W2 = np.random.randn(hidden, latent) * s
        self.Wd = np.random.randn(latent, dim)    * s
        self.mu  = np.zeros(latent)
        self.Li  = np.eye(latent)
        self.sigma_diag = np.ones(latent)  # for BN-style fallback
        self.tau = 1.0
        self.mu_d=0.; self.sd_d=1.; self.mu_e=0.; self.sd_e=1.; self.thr=1.

    def _sw(self, x): return x/(1+np.exp(-np.clip(x,-50,50)))

    def _enc(self, X):
        return self._sw(self._sw(X@self.W1)@self.W2)

    def _activate_normalise(self, H):
        """Apply activation + normalisation depending on ablation flags."""
        # ── Activation (Term 1) ───────────────────────────────────────────────
        if self.use_tcdf:
            T1 = H * _tcdf(H, self.nu)
        else:
            T1 = H * self._sw(H)  # plain Swish (sigmoid gate)

        # ── Amplifier (Term 2) ────────────────────────────────────────────────
        if self.use_amp and self.use_mcd:
            c  = H - self.mu
            w  = c @ self.Li.T
            m  = np.sqrt(np.maximum((w**2).sum(1), 0))
            gate = (m >= self.tau).astype(float)[:, None]
            amp  = self.alpha * np.sign(H) * (
                np.exp(np.clip(self.beta*(m[:,None]-self.tau),-20,20)) - 1)
            T2 = gate * amp
        elif self.use_amp and not self.use_mcd:
            # Euclidean-gated amplifier
            m    = np.linalg.norm(H, axis=1)
            gate = (m >= self.tau).astype(float)[:, None]
            amp  = self.alpha * np.sign(H) * (
                np.exp(np.clip(self.beta*(m[:,None]-self.tau),-20,20)) - 1)
            T2 = gate * amp
        else:
            T2 = np.zeros_like(T1)

        out = T1 + T2

        # ── Normalisation ─────────────────────────────────────────────────────
        if self.use_apn and self.use_mcd:
            c = out - self.mu
            return c @ self.Li.T    # Mahal. whitening
        elif not self.use_apn:
            # BN-style: centre by batch mean, scale by batch std
            mu_b  = out.mean(0)
            sd_b  = out.std(0) + 1e-6
            return (out - mu_b) / sd_b
        else:
            return out

    def _mcd(self, Z, h_frac=0.75):
        n = len(Z); h = max(int(n*h_frac), self.latent+1)
        best_det, best_mu, best_cov = np.inf, None, None
        for _ in range(6):
            idx = np.random.choice(n,h,replace=False); sub = Z[idx]
            for _ in range(5):
                mu  = sub.mean(0); d = sub-mu
                cov = d.T@d/max(len(sub)-1,1) + 1e-5*np.eye(self.latent)
                Si  = np.linalg.inv(cov)
                ds  = np.sqrt(np.maximum(
                    np.einsum('bi,ij,bj->b',Z-mu,Si,Z-mu),0))
                idx = np.argsort(ds)[:h]; sub = Z[idx]
            mu = sub.mean(0); d = sub-mu
            cov = d.T@d/max(len(sub)-1,1)
            det = np.linalg.det(cov+1e-5*np.eye(self.latent))
            if det < best_det:
                best_det=det; best_mu=mu; best_cov=cov
        return best_mu, best_cov

    def fit(self, X_train, n_epochs=60, lr=0.004, batch=64,
            lam1=0.5, m_in=1.0, m_out=3.0):
        lam1_eff = lam1 if self.use_cont else 0.0
        n = len(X_train)
        for ep in range(n_epochs):
            Z = self._enc(X_train)
            if self.use_mcd:
                mu, cov = self._mcd(Z)
                self.mu = mu
                try:
                    L = np.linalg.cholesky(cov+1e-5*np.eye(self.latent))
                    self.Li = np.linalg.inv(L)
                except np.linalg.LinAlgError:
                    self.Li = np.eye(self.latent)
            c = Z - self.mu; w = c @ self.Li.T
            dm = np.sqrt(np.maximum((w**2).sum(1),0))
            self.tau = self.momentum*self.tau + (1-self.momentum)*dm.mean()

            idx = np.random.permutation(n)
            for i in range(0, n, batch):
                xb = X_train[idx[i:i+batch]]
                zb = self._enc(xb)
                za = self._activate_normalise(zb)
                xh = za @ self.Wd
                err = xh - xb
                g   = np.clip(za.T@err/max(len(xb),1), -1, 1)
                self.Wd -= lr * g

        # calibrate
        Z  = self._enc(X_train)
        Za = self._activate_normalise(Z)
        Xh = Za @ self.Wd
        c  = Z - self.mu; w = c@self.Li.T
        dm = np.sqrt(np.maximum((w**2).sum(1),0))
        er = np.abs(X_train - Xh).mean(1)
        self.mu_d=dm.mean(); self.sd_d=max(dm.std(),1e-6)
        self.mu_e=er.mean(); self.sd_e=max(er.std(),1e-6)
        sd=np.maximum(0,(dm-self.mu_d)/self.sd_d)
        se=np.maximum(0,(er-self.mu_e)/self.sd_e)
        self.thr = np.percentile(0.5*sd+0.5*se, 95)
        return self

    def score(self, X):
        Z  = self._enc(X)
        Za = self._activate_normalise(Z)
        Xh = Za @ self.Wd
        c  = Z-self.mu; w=c@self.Li.T
        dm = np.sqrt(np.maximum((w**2).sum(1),0))
        er = np.abs(X-Xh).mean(1)
        sd = np.maximum(0,(dm-self.mu_d)/self.sd_d)
        se = np.maximum(0,(er-self.mu_e)/self.sd_e)
        return 0.5*sd+0.5*se

print("AblationModel defined")


## Cell 4 — Define ablation variants

In [ ]:
def make_variant(name, dim):
    """Factory: returns an AblationModel configured for the given variant."""
    base = dict(dim=dim, hidden=64, latent=32, alpha=1.0, beta=0.4, nu=4.0)
    variants = {
        "Deep-MELU (full)":       dict(**base),
        "No t-CDF (Swish base)":  dict(**base, use_tcdf=False),
        "No MCD (Euclidean)":     dict(**base, use_mcd=False),
        "No amplifier (α=0)":     dict(**base, use_amplifier=False),
        "No L_contrastive":       dict(**base, use_contrastive=False),
        "No APN-MCD (BN-style)":  dict(**base, use_apn_norm=False),
        "Ablate all":             dict(**base, use_tcdf=False, use_mcd=False,
                                       use_amplifier=False,
                                       use_contrastive=False,
                                       use_apn_norm=False),
    }
    return AblationModel(**variants[name])

VARIANTS = [
    "Deep-MELU (full)",
    "No t-CDF (Swish base)",
    "No MCD (Euclidean)",
    "No amplifier (α=0)",
    "No L_contrastive",
    "No APN-MCD (BN-style)",
    "Ablate all",
]

print(f"{len(VARIANTS)} ablation variants defined")


## Cell 5 — Run ablations

> **Expected runtime:** ~3–8 minutes (7 variants × 2 datasets)

In [ ]:
DATASETS_ABL = {
    "adversarial":  (X_adv, y_adv),
    "anisotropic":  (X_ani, y_ani),
}

abl_results = {}  # {dataset: {variant: metrics}}

scaler = StandardScaler()

for ds_name, (X, y) in DATASETS_ABL.items():
    print(f"\n{'='*55}")
    print(f"Dataset: {ds_name}")
    print(f"{'='*55}")
    X_sc = scaler.fit_transform(X)
    X_in = X_sc[y == 0]
    abl_results[ds_name] = {}

    for variant in VARIANTS:
        print(f"  {variant:<30}", end="", flush=True)
        try:
            m   = make_variant(variant, X.shape[1])
            m.fit(X_in, n_epochs=60)
            sc  = m.score(X_sc)
            thr = np.percentile(sc, 95)
            yh  = (sc > thr).astype(int)
            metrics = dict(
                auroc = float(roc_auc_score(y, sc)),
                aucpr = float(average_precision_score(y, sc)),
                f1    = float(f1_score(y, yh, zero_division=0)),
            )
            abl_results[ds_name][variant] = metrics
            delta = metrics["auroc"] - abl_results[ds_name].get(
                "Deep-MELU (full)", {}).get("auroc", metrics["auroc"])
            delta_str = f"  Δ={delta:+.3f}" if variant != "Deep-MELU (full)" else ""
            print(f"  AUROC={metrics['auroc']:.3f}  "
                  f"AUCPR={metrics['aucpr']:.3f}{delta_str}")
        except Exception as e:
            print(f"  ERROR: {e}")
            abl_results[ds_name][variant] = dict(auroc=0.5, aucpr=0., f1=0.)

print("\nAblation complete.")


## Cell 6 — Ablation table and interpretation

In [ ]:
rows = []
for ds_name, res in abl_results.items():
    full_auroc = res.get("Deep-MELU (full)", {}).get("auroc", 1.0)
    for variant, m in res.items():
        delta = m["auroc"] - full_auroc if variant != "Deep-MELU (full)" else 0.0
        rows.append({
            "Dataset":  ds_name,
            "Variant":  variant,
            "AUROC":    round(m["auroc"], 4),
            "AUCPR":    round(m["aucpr"], 4),
            "F1":       round(m["f1"],   4),
            "ΔAUROC":   round(delta, 4),
        })

df_abl = pd.DataFrame(rows)
pivot_abl = df_abl.pivot_table(
    index="Variant", columns="Dataset",
    values="AUROC").round(4)

# Sort by average descending
pivot_abl["Average"] = pivot_abl.mean(axis=1)
pivot_abl = pivot_abl.sort_values("Average", ascending=False)

display(pivot_abl.style
        .background_gradient(cmap="YlGn",
                             subset=list(DATASETS_ABL.keys()))
        .format("{:.4f}")
        .set_caption("Ablation AUROC — adversarial & anisotropic"))

# Interpretation
print("\n--- Component contribution analysis ---")
full = {ds: abl_results[ds]["Deep-MELU (full)"]["auroc"]
        for ds in DATASETS_ABL}
for variant in VARIANTS[1:]:
    impacts = []
    for ds in DATASETS_ABL:
        delta = abl_results[ds][variant]["auroc"] - full[ds]
        impacts.append(delta)
    avg_impact = np.mean(impacts)
    direction  = "HURTS" if avg_impact < -0.005 else (
                 "helps" if avg_impact >  0.005 else "neutral")
    print(f"  Removing '{variant[3:]:30s}': avg Δ={avg_impact:+.4f}  [{direction}]")


## Cell 7 — Ablation visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Deep-MELU Ablation Study — Adversarial & Anisotropic Datasets",
             fontsize=13)

VARIANT_COLORS = {
    "Deep-MELU (full)":      "#1D9E75",
    "No t-CDF (Swish base)": "#D85A30",
    "No MCD (Euclidean)":    "#534AB7",
    "No amplifier (α=0)":   "#BA7517",
    "No L_contrastive":      "#888780",
    "No APN-MCD (BN-style)": "#E24B4A",
    "Ablate all":            "#2C2C2A",
}

for ax_idx, (ds_name, _) in enumerate(DATASETS_ABL.items()):
    ax = axes[ax_idx]
    variant_names = list(VARIANTS)
    aurocs = [abl_results[ds_name][v]["auroc"] for v in variant_names]
    full_auroc = abl_results[ds_name]["Deep-MELU (full)"]["auroc"]
    colors = [VARIANT_COLORS[v] for v in variant_names]

    bars = ax.barh(variant_names, aurocs, color=colors, alpha=0.87)
    ax.axvline(full_auroc, color="#1D9E75", lw=1.5, ls="--", alpha=0.6,
               label=f"Full model ({full_auroc:.3f})")
    ax.axvline(0.5, color="gray", lw=0.8, ls=":", alpha=0.5)

    for bar, val, variant in zip(bars, aurocs, variant_names):
        delta = val - full_auroc
        label = f"{val:.3f}"
        if variant != "Deep-MELU (full)":
            label += f" ({delta:+.3f})"
        ax.text(max(val, 0.3) + 0.005, bar.get_y() + bar.get_height()/2,
                label, va="center", fontsize=9,
                color="#712B13" if delta < -0.01 else "#085041")

    ax.set_xlabel("AUROC"); ax.set_xlim(0.3, 1.1)
    ax.set_title(f"Dataset: {ds_name}", fontsize=11)
    ax.legend(fontsize=9); ax.grid(axis="x", alpha=0.3)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig("outputs/ablation_fixed.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → outputs/ablation_fixed.png")

df_abl.to_csv("outputs/ablation_results.csv", index=False)
print("CSV saved → outputs/ablation_results.csv")
